In [ ]:
# FASE 4 — MODELING | CardioRisk · IBM Data Science · CRISP-DM
# Objetivo: entrenar clasificadores optimizados para Recall (minimizar falsos negativos)

# ── Instalaciones ──
!pip install kagglehub imbalanced-learn xgboost --quiet

import kagglehub, os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0f1a'
matplotlib.rcParams['axes.facecolor']   = '#0d1526'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#7a8fa8'
matplotlib.rcParams['ytick.color']      = '#7a8fa8'
matplotlib.rcParams['axes.edgecolor']   = '#1a2c3d'
matplotlib.rcParams['grid.color']       = '#1a2c3d'
warnings.filterwarnings('ignore')
print("✓ Librerías cargadas")

In [ ]:
# ── BLOQUE 1: CARGA Y PREPROCESAMIENTO (autocontenido) ──
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Descargar dataset
KAGGLE_DATASET = "jocelyndumlao/cardiovascular-disease-dataset"
try:
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    csv_path = next(f for f in [os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs] if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
    print(f"✓ Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas")
except Exception as e:
    print(f"⚠ kagglehub: {e}\n→ Sube manualmente 'cardiovascular_disease_dataset.csv' a /content/")
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')

# Limpieza básica
df.columns = df.columns.str.lower().str.strip()
df = df.dropna()

# Features y target
CATEGORICAL = ['gender','chestpain','restingelectro']
NUMERICAL   = ['age','restingBP','serumcholestrol','maxheartrate','oldpeak','noofmajorvessels']
TARGET      = 'target'

# Encoding categórico
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)
feature_cols = [c for c in df_enc.columns if c != TARGET]

X = df_enc[feature_cols]
y = df_enc[TARGET]

# Split 70/15/15 estratificado
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# SMOTE en train
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

print(f"Train: {X_train_sm.shape[0]} | Val: {X_val_sc.shape[0]} | Test: {X_test_sc.shape[0]}")
print(f"Clases train tras SMOTE: {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")
print(f"Features: {X_train_sm.shape[1]}")

In [ ]:
# ── BLOQUE 2: ENTRENAMIENTO DE CLASIFICADORES ──
# NOTA: NO se usa class_weight='balanced' porque SMOTE ya balancea el train set
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', verbosity=0),
    "SVM":                 SVC(probability=True, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_val_sc)
    y_prob = model.predict_proba(X_val_sc)[:,1]
    results[name] = {
        "Recall":    recall_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "F1":        f1_score(y_val, y_pred),
        "AUC-ROC":   roc_auc_score(y_val, y_prob),
        "Accuracy":  accuracy_score(y_val, y_pred),
        "model":     model
    }
    print(f"{name:<25} Recall={results[name]['Recall']:.3f}  AUC={results[name]['AUC-ROC']:.3f}  F1={results[name]['F1']:.3f}")

In [ ]:
# ── BLOQUE 3: COMPARACIÓN VISUAL ──
df_res = pd.DataFrame({k:{m:v for m,v in vals.items() if m!='model'} for k,vals in results.items()}).T
df_res = df_res.sort_values('Recall', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['Recall','AUC-ROC','F1']
colors  = ['#00f2fe','#8b5cf6','#00ff88']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.barh(df_res.index, df_res[metric], color=color, alpha=.75, edgecolor='none')
    ax.set_xlim(0, 1.05)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.axvline(0.8, color='#ff9500', linestyle='--', alpha=.5, linewidth=1)
    for bar, val in zip(bars, df_res[metric]):
        ax.text(val + .01, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va='center', fontsize=9)
    ax.grid(axis='x', alpha=.3)

plt.suptitle('Comparación de Modelos — Validación', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/f4_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
plt.show()

print("\n📊 RANKING POR RECALL:")
print(df_res[['Recall','AUC-ROC','F1','Precision']].to_string())

In [ ]:
# ── BLOQUE 4: OPTIMIZACIÓN DEL MEJOR MODELO ──
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer

# Seleccionar el modelo con mayor Recall en validación
best_name = df_res['Recall'].idxmax()
print(f"Modelo seleccionado: {best_name} (Recall val: {df_res.loc[best_name,'Recall']:.3f})")

recall_scorer = make_scorer(recall_score)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grids de hiperparámetros por modelo
param_grids = {
    "Random Forest":     {"n_estimators":[200,400],"max_depth":[None,8,15],"min_samples_split":[2,5]},
    "XGBoost":           {"n_estimators":[100,200,300],"max_depth":[3,5,7],"learning_rate":[.05,.1]},
    "Logistic Regression":{"C":[0.1,1,10],"solver":["lbfgs","saga"]},
    "Gradient Boosting": {"n_estimators":[100,200],"max_depth":[3,5],"learning_rate":[.05,.1]},
    "SVM":               {"C":[0.1,1,10],"kernel":["rbf","linear"]},
}

base_model = models[best_name].__class__(**{k:v for k,v in models[best_name].get_params().items()})

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grids.get(best_name, {}),
    scoring=recall_scorer,
    cv=cv, n_jobs=-1, verbose=0
)
grid.fit(X_train_sm, y_train_sm)

best_model = grid.best_estimator_
y_pred_val = best_model.predict(X_val_sc)
y_prob_val = best_model.predict_proba(X_val_sc)[:,1]

print(f"\nMejores hiperparámetros: {grid.best_params_}")
print(f"Recall  CV:  {grid.best_score_:.3f}")
print(f"Recall  Val: {recall_score(y_val, y_pred_val):.3f}")
print(f"AUC-ROC Val: {roc_auc_score(y_val, y_prob_val):.3f}")
print(f"F1      Val: {f1_score(y_val, y_pred_val):.3f}")

In [ ]:
# ── BLOQUE 5: CURVA ROC + GUARDAR MODELO ──
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_val, y_prob_val)
auc = roc_auc_score(y_val, y_prob_val)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#00f2fe', lw=2, label=f'{best_name} (AUC = {auc:.3f})')
ax.plot([0,1],[0,1], color='#3a5570', lw=1, linestyle='--', label='Random (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=.08, color='#00f2fe')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title(f'Curva ROC — {best_name}', fontsize=13)
ax.legend(loc='lower right')
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('/content/f4_roc_curve.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
plt.show()

# Guardar artefactos
joblib.dump(best_model, '/content/best_model.pkl')
joblib.dump(scaler,     '/content/scaler.pkl')
joblib.dump(feature_cols, '/content/feature_cols.pkl')

print(f"\n✓ Modelo guardado: /content/best_model.pkl")
print(f"✓ Scaler guardado:  /content/scaler.pkl")
print(f"✓ Features:         /content/feature_cols.pkl")
print(f"\n→ FASE 4 COMPLETA — Modelo: {best_name}")